# Gold Layer — Revenue by State

Aggregates total revenue per Brazilian state. Uses a three-table join across `silver.order_items`, `silver.orders`, and `silver.customers` — items hold revenue, orders bridge items to customers via order_id, and customers hold the state attribute.

## Design Decisions

* **Revenue defined as `SUM(price + freight_value)` per row** — same interpretation as `gold.monthly_revenue`. Treated each order_items row as a distinct unit with its own freight allocation.
* **Status filter on orders** — excludes canceled and unavailable orders, consistent with the rest of the Gold layer.
* **Three-table chained join** — order_items → orders → customers. Selected only the columns needed from each Silver table before joining to keep DataFrames small.
* **Expected output** — Brazil has 27 states (26 plus Federal District). The table should contain ~27 rows; missing or extra rows would indicate a broken join.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'
quarantine_schema = 'quarantine'

### Revenue By State

In [0]:

df_orders_state = spark.read.table(f'{catalog_name}.{silver_schema}.orders')\
    .filter(~col('order_status').isin(['canceled', 'unavailable']))\
        .select('order_id','customer_id')

df_orders_state.createOrReplaceTempView('order_st')
                                                                                                                                                        
df_order_items_state = spark.read.table(f'{catalog_name}.{silver_schema}.order_items').select('order_id','price','freight_value')
df_order_items_state.createOrReplaceTempView('order_items_st')

df_customers_state = spark.read.table(f'{catalog_name}.{silver_schema}.customers').select('customer_id','customer_state')
df_customers_state.createOrReplaceTempView('customers_st')


In [0]:
revenue_by_state_df = spark.sql('''Select c.customer_state as state,sum(t.price + t.freight_value) as total_revenue
from order_items_st t inner join order_st o on t.order_id = o.order_id
inner join customers_st c on o.customer_id = c.customer_id
group by c.customer_state''')

In [0]:
check_not_null(revenue_by_state_df, ['state', 'total_revenue'])
check_unique(revenue_by_state_df, ['state'])
check_value_range(revenue_by_state_df, [{'column': 'total_revenue', 'min_value': 0, 'inclusive': True}])

check_not_null: PASSED
check_unique: PASSED
check_value_range: PASSED


In [0]:
revenue_by_state_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{gold_schema}.revenue_by_state')